# PaddleOCR-VL-1.6 — baseline toàn trang trên Colab T4

Dùng **pipeline chính thức**: PP-DocLayoutV3 phát hiện/sắp xếp vùng,
PaddleOCR-VL-1.6 nhận dạng nội dung. Backend **Transformers**, VLM FP16, layout FP32.
Không dùng ví dụ OCR một vùng để thay thế pipeline phân tích toàn trang.

Nhận đúng 43 ảnh từ GitHub, kiểm tra hash; không tải ground truth.
Đầu ra ZIP: `outputs/paddleocr_vl_1_6_colab_t4_fp16/{predictions.jsonl,run.json,texts/}`.
Kết quả native/HTML và số token từng lần sinh được lưu trong JSONL.
Text được ghép theo block, chuyển bảng HTML thành text để chấm bằng runner local.

[Model 1.6](https://huggingface.co/PaddlePaddle/PaddleOCR-VL-1.6),
[layout Transformers chính thức](https://huggingface.co/PaddlePaddle/PP-DocLayoutV3_safetensors),
[hướng dẫn pipeline và T4](https://www.paddleocr.ai/latest/en/version3.x/pipeline_usage/PaddleOCR-VL.html).


## 1. Môi trường riêng trên Colab T4

Chọn **T4 GPU** và chạy từ trên xuống trong một session riêng, sau khi đã lưu kết quả mô hình trước.
Không cài đè notebook này vào session Qwen đang chạy: phiên bản Transformers khác nhau.
Giữ PyTorch/CUDA của Colab, không cài requirements OCR local.
Nếu đã import Transformers trước ô cài thư viện, restart session sau khi cài rồi chạy tiếp từ ô cấu hình.


In [ ]:
import subprocess
import sys
PINS = ['transformers==5.3.0', 'accelerate==1.12.0', 'huggingface-hub==1.5.0', 'Pillow==11.3.0', 'requests>=2.32,<3', 'paddleocr[doc-parser]==3.7.0', 'paddlex==3.7.2', 'beautifulsoup4==4.14.2', 'sentencepiece==0.2.1']
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PINS])
print("Đã cài thư viện. Tiếp tục với ô cấu hình.")


In [ ]:
from pathlib import Path
REPO_URL = "https://github.com/datpq-alpha/ocr"
DATA_COMMIT = "eab241b9cd2c43c81255c247d81c49e83bcca04a"
EXPECTED_MANIFEST_SHA256 = "f64b2d370874e49e949b043776ab6a0db4f04399365aedb572a072b37c29dc68"
WARMUP_NEW_TOKENS = 64
SEED = 42
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_ROOT = Path("/content/drive/MyDrive/ocr_baseline")
else:
    WORK_ROOT = Path("/content/ocr_baseline")
DATA_ROOT = Path("/content/ocr_baseline_data")

MODEL_ID = "PaddlePaddle/PaddleOCR-VL-1.6"
MODEL_REVISION = "c5630abae1d940eafe0697512a0325494b02ab42"
LAYOUT_MODEL_ID = "PaddlePaddle/PP-DocLayoutV3_safetensors"
LAYOUT_REVISION = "e4f489dc536556fc1be02b973ba06756e0d4a2ac"
ENGINE = "paddleocr_vl_1_6"
MIN_PIXELS = 144 * 28 * 28
MAX_PIXELS = 1280 * 28 * 28
MAX_NEW_TOKENS = 4096  # Per detected region, not a shared page budget.
RUN_NAME = "paddleocr_vl_1_6_colab_t4_fp16"
OUTPUT_DIR = WORK_ROOT / "outputs" / RUN_NAME
PROMPT = "Native PaddleOCR-VL-1.6 task prompts by layout label (OCR/Table/Seal)"
assert 0 < MIN_PIXELS <= MAX_PIXELS and MAX_NEW_TOKENS > 0
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in (".", "..")
print("Output:", OUTPUT_DIR)


In [ ]:
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import re
import time
import traceback
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import PurePosixPath


def canonical_sha256(value):
    encoded = json.dumps(value, ensure_ascii=False, sort_keys=True,
                         separators=(",", ":"), allow_nan=False).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def atomic_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp-" + uuid.uuid4().hex)
    try:
        with temporary.open("w", encoding="utf-8", newline="\n") as stream:
            stream.write(text)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def atomic_json(path, value):
    atomic_text(path, json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n")


def percentile(values, proportion):
    if not values:
        return None
    ordered = sorted(values)
    position = (len(ordered) - 1) * proportion
    lower, upper = math.floor(position), math.ceil(position)
    return ordered[lower] + (ordered[upper] - ordered[lower]) * (position - lower)


def validate_samples(manifest):
    ids, paths = set(), set()
    samples = manifest["samples"]
    if not samples:
        raise ValueError("Empty manifest")
    for sample in samples:
        key, relative = sample["sample_id"], sample["image_path"]
        path = PurePosixPath(relative)
        if not re.fullmatch(r"GT\d{6}", key) or key in ids:
            raise ValueError(f"Invalid/duplicate ID: {key}")
        if (path.is_absolute() or ".." in path.parts or "\\" in relative
                or len(path.parts) < 3 or path.parts[0] != "Accounting_Documents"
                or path.parts[1] not in ("02_Accounting_Vouchers", "03_Invoices")
                or relative in paths):
            raise ValueError(f"Invalid/duplicate image path: {relative}")
        if not re.fullmatch(r"[0-9a-f]{64}", sample["sha256"]):
            raise ValueError(f"Invalid image hash: {key}")
        ids.add(key)
        paths.add(relative)
    return samples


def validate_row(sample, row, fingerprint):
    if (row.get("sample_id") != sample["sample_id"]
            or row.get("image_path") != sample["image_path"]
            or row.get("image_sha256") != sample["sha256"]
            or row.get("run_fingerprint") != fingerprint):
        raise ValueError(f"Saved prediction identity mismatch: {sample['sample_id']}")
    if row.get("status") not in ("Success", "Failed"):
        raise ValueError("Invalid prediction status")
    if row["status"] == "Success":
        if not isinstance(row.get("text"), str) or row["timing_seconds"]["total"] < 0:
            raise ValueError("Invalid text/timing")


def load_records(output_dir, samples, fingerprint):
    records = {}
    expected = {s["sample_id"]: s for s in samples}
    for path in (output_dir / "records").glob("*.json"):
        if path.stem not in expected:
            raise ValueError(f"Unexpected saved sample: {path.name}")
        row = json.loads(path.read_text(encoding="utf-8"))
        validate_row(expected[path.stem], row, fingerprint)
        records[path.stem] = row
    return records


def write_artifacts(output_dir, samples, records, metadata):
    rows = [records[s["sample_id"]] for s in samples if s["sample_id"] in records]
    atomic_text(output_dir / "predictions.jsonl", "".join(
        json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n" for row in rows))
    for row in rows:
        if row["status"] == "Success":
            text_path = output_dir / "texts" / (row["sample_id"] + ".txt")
            if not text_path.exists():
                atomic_text(text_path, row["text"] + "\n")
            elif text_path.read_text(encoding="utf-8") != row["text"] + "\n":
                raise ValueError(f"Saved text was edited: {text_path}; use a new RUN_NAME")
    successful = [row for row in rows if row["status"] == "Success"]
    times = [row["timing_seconds"]["total"] for row in successful]
    failed = len(rows) - len(successful)
    metadata.update(
        completed_images=len(successful), failed_images=failed,
        pending_images=len(samples) - len(rows),
        failure_rate=failed / len(rows) if rows else 0.0,
        truncated_images=sum(row.get("truncated", False) for row in successful),
        latency_p50_seconds=percentile(times, 0.5),
        latency_p95_seconds=percentile(times, 0.95),
        mean_image_seconds=sum(times) / len(times) if times else None,
        wall_seconds=sum(s["wall_seconds"] for s in metadata.get("sessions", [])),
        initialization_seconds=sum(s["initialization_seconds"] for s in metadata.get("sessions", [])),
        warmup_seconds=sum(s["warmup_seconds"] for s in metadata.get("sessions", [])),
        peak_gpu_allocated_bytes=max((r.get("gpu_memory", {}).get("peak_allocated_bytes", 0)
                                      for r in rows), default=0),
        peak_gpu_reserved_bytes=max((r.get("gpu_memory", {}).get("peak_reserved_bytes", 0)
                                     for r in rows), default=0),
    )
    wall = metadata["wall_seconds"]
    metadata["throughput_images_per_minute"] = len(successful) * 60 / wall if wall else None
    atomic_json(output_dir / "run.json", metadata)


def run_benchmark(samples, output_dir, identity, session_info, predict_fn):
    """Checkpoint each image; resume missing IDs without mixing configurations."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "records").mkdir(exist_ok=True)
    (output_dir / "texts").mkdir(exist_ok=True)
    fingerprint = canonical_sha256(identity)
    run_path = output_dir / "run.json"
    if run_path.exists():
        metadata = json.loads(run_path.read_text(encoding="utf-8"))
        if metadata.get("run_fingerprint") != fingerprint:
            raise ValueError("Configuration/data/runtime changed; choose a new RUN_NAME")
    else:
        if any((output_dir / "records").iterdir()) or (output_dir / "predictions.jsonl").exists():
            raise ValueError("Existing output without provenance; choose a new RUN_NAME")
        metadata = dict(identity)
        metadata.update(created_at=datetime.now(timezone.utc).isoformat(),
                        run_fingerprint=fingerprint, sessions=[])
    records = load_records(output_dir, samples, fingerprint)
    # Rebuild the exported JSONL from atomic per-image records after interruptions.
    write_artifacts(output_dir, samples, records, metadata)
    pending = [s for s in samples if s["sample_id"] not in records]
    if not pending:
        print("No missing images. Existing results retained.")
        return metadata
    session = dict(session_info, session_id=uuid.uuid4().hex,
                   started_at=datetime.now(timezone.utc).isoformat(), wall_seconds=0.0,
                   attempted_images=0, status="Running")
    metadata["sessions"].append(session)
    metadata["status"] = "Running"
    started = time.perf_counter()
    try:
        write_artifacts(output_dir, samples, records, metadata)
        for sample in pending:
            print(f"[START {len(records) + 1}/{len(samples)}] {sample['sample_id']}", flush=True)
            row = {"sample_id": sample["sample_id"], "image_path": sample["image_path"],
                   "image_sha256": sample["sha256"], "engine": identity["engine"],
                   "run_fingerprint": fingerprint, "session_id": session["session_id"]}
            attempt_started = time.perf_counter()
            try:
                row.update(predict_fn(sample), status="Success")
            except Exception:
                row.update(status="Failed", error=traceback.format_exc(),
                           timing_seconds={"total": time.perf_counter() - attempt_started})
                import gc
                gc.collect()
                if "torch" in globals() and torch.cuda.is_available():
                    torch.cuda.empty_cache()
            validate_row(sample, row, fingerprint)
            print(f"  {sample['sample_id']}: OCR {row['status']}; saving checkpoint"
                  f" ({row['timing_seconds']['total']:.1f}s"
                  + (f", {row['generated_tokens']} tokens" if "generated_tokens" in row else "")
                  + ")", flush=True)
            atomic_json(output_dir / "records" / (sample["sample_id"] + ".json"), row)
            records[sample["sample_id"]] = row
            session["attempted_images"] += 1
            session["wall_seconds"] = time.perf_counter() - started
            write_artifacts(output_dir, samples, records, metadata)
            print(f"[{len(records)}/{len(samples)}] {sample['sample_id']}: {row['status']}"
                  + (" (TOKEN LIMIT)" if row.get("truncated") else ""), flush=True)
    finally:
        session["wall_seconds"] = time.perf_counter() - started
        session["status"] = "Completed" if len(records) == len(samples) else "Interrupted"
        metadata["status"] = ("Interrupted" if len(records) < len(samples) else
                              "CompletedWithErrors" if any(r["status"] == "Failed" for r in records.values())
                              else "Completed")
        write_artifacts(output_dir, samples, records, metadata)
    return metadata


## 2. Ảnh từ GitHub

Cố định commit và fingerprint của manifest đang khớp với máy local. Mỗi ảnh được kiểm tra
SHA-256 trước khi nhận dạng. Chỉ tải manifest và ảnh, không tải texts/annotations của ground truth.
Khác biệt CRLF/LF của manifest không ảnh hưởng fingerprint nội dung JSON.


In [ ]:
from urllib.parse import quote
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

http = requests.Session()
http.mount("https://", HTTPAdapter(max_retries=Retry(
    total=4, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"])))
raw_base = f"https://raw.githubusercontent.com/datpq-alpha/ocr/{DATA_COMMIT}/"
response = http.get(raw_base + "ground_truth/manifest.json", timeout=(15, 90))
response.raise_for_status()
manifest = response.json()
if canonical_sha256(manifest) != EXPECTED_MANIFEST_SHA256:
    raise ValueError("Manifest differs from the local baseline; check DATA_COMMIT/fingerprint")
samples = validate_samples(manifest)
if len(samples) != 43:
    raise ValueError("Expected the same 43 images as the local baseline")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
atomic_json(DATA_ROOT / "manifest.json", manifest)
for sample in samples:
    destination = DATA_ROOT / sample["image_path"]
    if not destination.exists() or file_sha256(destination) != sample["sha256"]:
        response = http.get(raw_base + quote(sample["image_path"], safe="/"), timeout=(15, 90))
        response.raise_for_status()
        if hashlib.sha256(response.content).hexdigest() != sample["sha256"]:
            raise ValueError(f"Downloaded image hash mismatch: {sample['sample_id']}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(response.content)
    if file_sha256(destination) != sample["sha256"]:
        raise ValueError(f"Image hash mismatch: {sample['sample_id']}")
print("Verified images:", len(samples), dict(Counter(s["category"] for s in samples)))


## 3. Pipeline toàn trang: PP-DocLayoutV3 + PaddleOCR-VL-1.6

Dùng `PaddleOCRVL(pipeline_version="v1.6", engine="transformers")` chính thức.
Layout chạy FP32; VLM 0,9B chạy FP16/SDPA trên T4, không lượng tử hóa.
Cả hai checkpoint được tải ở revision cố định. Không cài PaddlePaddle GPU, vLLM, BF16 hoặc FlashAttention 2.

Pipeline nhận dạng các vùng chữ/bảng/dấu và giữ thứ tự đọc của kết quả native.
Giữ mọi block, kể cả header/footer. Nội dung bảng HTML được chuyển thành text theo thứ tự hàng/ô,
không sửa chữ/số; HTML và kết quả native vẫn được lưu để đối chiếu.
CER/WER đo chuỗi văn bản sau bước chuyển này, chưa chấm cấu trúc bảng.


In [ ]:
import gc
import random
import torch
from PIL import Image, ImageOps
import transformers
if transformers.__version__ != "5.3.0":
    raise RuntimeError("Restart session sau ô cài thư viện; cần Transformers 5.3.0.")
if not torch.cuda.is_available():
    raise RuntimeError("Chọn T4 GPU trong Runtime → Change runtime type.")
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.backends.cuda.matmul.allow_tf32 = False
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}; VRAM: {gpu.total_memory / 2**30:.2f} GiB; CUDA: {torch.version.cuda}")
if "T4" not in gpu.name:
    print("GPU khác T4; phần cứng thực tế được lưu để báo cáo riêng.")

import copy
import numpy as np
import yaml
import paddlex
from paddleocr import PaddleOCRVL
from huggingface_hub import snapshot_download

torch.cuda.synchronize()
load_started = time.perf_counter()
allow_files = ["*.json", "*.safetensors", "*.model", "*.jinja", "*.yml"]
vl_directory = snapshot_download(MODEL_ID, revision=MODEL_REVISION, allow_patterns=allow_files)
layout_directory = snapshot_download(LAYOUT_MODEL_ID, revision=LAYOUT_REVISION,
                                     allow_patterns=allow_files)
config_path = Path(paddlex.__file__).parent / "configs/pipelines/PaddleOCR-VL-1.6.yaml"
pipeline_config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
pipeline_config.update(
    batch_size=1, use_queues=False, use_doc_preprocessor=False,
    use_layout_detection=True, use_chart_recognition=False, use_seal_recognition=True,
    use_ocr_for_image_block=True, format_block_content=False, markdown_ignore_labels=[])
layout_config = pipeline_config["SubModules"]["LayoutDetection"]
layout_config.update(model_dir=layout_directory, batch_size=1,
                     engine="transformers",
                     engine_config={"dtype": "float32", "trust_remote_code": False})
vl_config = pipeline_config["SubModules"]["VLRecognition"]
vl_config.update(model_dir=vl_directory, batch_size=1, engine="transformers",
                 engine_config={"dtype": "float16", "attn_implementation": "sdpa",
                                "trust_remote_code": False})
if "pipeline" in globals():
    del pipeline
gc.collect()
torch.cuda.empty_cache()
pipeline = PaddleOCRVL(pipeline_version="v1.6", engine="transformers", device="gpu:0",
                       paddlex_config=pipeline_config, use_queues=False,
                       use_doc_orientation_classify=False, use_doc_unwarping=False)
# Pinned PaddleX exposes the native Hugging Face VLM here; inspect before benchmarking.
model = pipeline.paddlex_pipeline.vl_rec_model.infer
processor = pipeline.paddlex_pipeline.vl_rec_model.processor
if not isinstance(model, torch.nn.Module) or model.device.type != "cuda":
    raise RuntimeError("Expected a local Transformers VLM on CUDA")
torch.cuda.synchronize()
initialization_seconds = time.perf_counter() - load_started
packages = {name: importlib.metadata.version(name) for name in
            ("torch", "torchvision", "transformers", "accelerate", "huggingface-hub",
             "Pillow", "requests", "paddleocr", "paddlex", "beautifulsoup4", "sentencepiece")}
configuration = {
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "layout_model_id": LAYOUT_MODEL_ID, "layout_revision": LAYOUT_REVISION,
    "pipeline_version": "v1.6", "backend": "transformers",
    "quantization": "none", "compute_dtype": "float16", "layout_dtype": "float32",
    "attention_backend": "sdpa", "prompt": PROMPT,
    "max_new_tokens_per_region": MAX_NEW_TOKENS, "min_pixels": MIN_PIXELS,
    "max_pixels": MAX_PIXELS, "use_queues": False, "use_cache": True,
    "preprocessing": "EXIF transpose; alpha on white; RGB/BGR conversion; native layout and region resize",
    "reading_order": "native PaddleOCRVL parsing_res_list order; no header/footer exclusion",
    "text_assembly": "table HTML -> rows/cells via HTMLParser; preserve non-table content",
    "layout_threshold": layout_config.get("threshold"),
    "use_seal_recognition": True, "use_ocr_for_image_block": True,
    "layout_shape_mode": "rect",
}

configuration["prompt_sha256"] = hashlib.sha256(PROMPT.encode("utf-8")).hexdigest()
configuration.update(batch_size=1, seed=SEED, do_sample=False, num_beams=1,
                     warmup_images_per_session=1, warmup_max_new_tokens=WARMUP_NEW_TOKENS,
                     measured_inference_passes_per_image=1,
                     cpu_threads=torch.get_num_threads())
identity = {
    "engine": ENGINE, "device": "cuda:0", "python": sys.version,
    "platform": platform.platform(), "packages": packages, "configuration": configuration,
    "hardware": {"gpu": gpu.name, "gpu_total_memory_bytes": gpu.total_memory,
                 "compute_capability": list(torch.cuda.get_device_capability(0)),
                 "cuda": torch.version.cuda, "cpu_logical_count": os.cpu_count()},
    "source": {"repository_url": REPO_URL, "commit": DATA_COMMIT,
               "manifest_sha256": EXPECTED_MANIFEST_SHA256},
    "samples": [{k: s[k] for k in ("sample_id", "image_path", "sha256")} for s in samples],
    "timing_scope": "full image pipeline including decode/preprocess/GPU/generation/assembly; CUDA synchronized; excludes downloads/model load/warmup",
    "truncation_policy": "retain partial text as Success with truncated=true; include in CER/WER",
    "measured_inference_passes_per_image": 1,
    "initialization_includes_uncached_model_download": True,
}
print("Initialization:", round(initialization_seconds, 2), "seconds")


In [ ]:
import threading


class ImageProgress:
    """Only observe progress; never change generated tokens or stopping limits."""
    def __init__(self, sample_id, interval=30):
        self.sample_id = sample_id
        self.interval = interval
        self.stage = "starting"
        self.generated_tokens = 0
        self.last_token_at = None
        self.stop_event = threading.Event()

    def update(self, stage, generated_tokens=None):
        self.stage = stage
        if generated_tokens is not None and generated_tokens > self.generated_tokens:
            self.generated_tokens = generated_tokens
            self.last_token_at = time.perf_counter()

    def token_observer(self, prompt_tokens):
        def observe(input_ids, scores, **kwargs):
            self.update("generation", int(input_ids.shape[-1]) - prompt_tokens)
            return False  # Observe only: EOS and max_new_tokens still decide when to stop.
        return observe

    def emit(self):
        elapsed = time.perf_counter() - self.started
        suffix = (f"; last new token {time.perf_counter() - self.last_token_at:.0f}s ago"
                  if self.last_token_at is not None else "; waiting for first generated token")
        print(f"  [{self.sample_id}] {elapsed:.0f}s; stage={self.stage}; "
              f"generated={self.generated_tokens} tokens" + suffix, flush=True)

    def _watch(self):
        while not self.stop_event.wait(self.interval):
            self.emit()

    def __enter__(self):
        self.started = time.perf_counter()
        self.thread = threading.Thread(target=self._watch, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, exc_type, exc_value, exc_tb):
        self.stop_event.set()
        self.thread.join(timeout=1)



from contextlib import contextmanager
from html.parser import HTMLParser


class TableText(HTMLParser):
    """Remove only HTML structure, inserting boundaries to avoid joining words."""
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.parts = []
        self.ignored = 0

    def handle_starttag(self, tag, attrs):
        if tag in ("script", "style"):
            self.ignored += 1
        elif not self.ignored and tag in ("br", "tr", "p"):
            self.parts.append("\n")

    def handle_endtag(self, tag):
        if tag in ("script", "style"):
            self.ignored = max(0, self.ignored - 1)
        elif not self.ignored and tag in ("td", "th"):
            self.parts.append("\t")
        elif not self.ignored and tag in ("tr", "p"):
            self.parts.append("\n")

    def handle_data(self, data):
        if not self.ignored:
            self.parts.append(data)


def block_to_text(block):
    content = block.get("block_content", "")
    if not isinstance(content, str):
        raise TypeError("Native block_content must be text")
    if block.get("block_label") == "table" and re.search(r"<(?:table|tr|td|th)\b", content, re.I):
        parser = TableText()
        parser.feed(content)
        parser.close()
        return "".join(parser.parts).strip()
    return content


@contextmanager
def observe_region_generation(model, processor, progress, calls):
    """Scoped observer for the pinned pipeline; restore generate even on interruption."""
    original_generate = model.generate
    had_instance_override = "generate" in model.__dict__
    instance_override = model.__dict__.get("generate")

    def generate(*args, **kwargs):
        ids = kwargs.get("input_ids")
        if ids is None or ids.shape[0] != 1:
            raise RuntimeError("Expected region batch size 1")
        prompt_tokens = int(ids.shape[-1])
        offset = sum(call["generated_tokens"] for call in calls)
        region_number = len(calls) + 1
        progress.update(f"region {region_number}: generation / first-token prefill")
        def observe(input_ids, scores, **extra):
            count = int(input_ids.shape[-1]) - prompt_tokens
            progress.update(f"region {region_number}: generation", offset + count)
            return False
        criteria = list(kwargs.get("stopping_criteria") or [])
        kwargs.update(stopping_criteria=criteria + [observe],
                      do_sample=False, num_beams=1, use_cache=True,
                      return_dict_in_generate=False)
        generated = original_generate(*args, **kwargs)
        token_ids = generated[0, prompt_tokens:].detach().cpu().tolist()
        eos = model.generation_config.eos_token_id
        eos = {eos} if isinstance(eos, int) else set(eos or [])
        limit = int(kwargs["max_new_tokens"])
        truncated = len(token_ids) >= limit and (not token_ids or token_ids[-1] not in eos)
        calls.append({
            "region_call": region_number, "input_tokens": prompt_tokens,
            "generated_tokens": len(token_ids), "truncated": truncated,
            "raw_text": processor.tokenizer.decode(
                token_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False),
        })
        progress.update("layout / region postprocess", offset + len(token_ids))
        return generated

    model.generate = generate
    try:
        yield
    finally:
        if had_instance_override:
            model.generate = instance_override
        else:
            del model.generate


def predict_image(sample, max_new_tokens=MAX_NEW_TOKENS):
    with ImageProgress(sample["sample_id"]) as progress:
        progress.update("verify image / GPU sync")
        image_path = DATA_ROOT / sample["image_path"]
        if file_sha256(image_path) != sample["sha256"]:
            raise ValueError("Image changed after download")
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        with Image.open(image_path) as opened:
            rgba = ImageOps.exif_transpose(opened).convert("RGBA")
            white = Image.new("RGBA", rgba.size, "white")
            white.alpha_composite(rgba)
            image = white.convert("RGB")
        image_bgr = np.asarray(image)[:, :, ::-1].copy()
        decoded = time.perf_counter()
        progress.update("layout detection / region preprocess")
        calls = []
        with observe_region_generation(model, processor, progress, calls):
            pages = list(pipeline.predict(
                image_bgr, use_queues=False, layout_shape_mode="rect",
                min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS, max_new_tokens=max_new_tokens,
                markdown_ignore_labels=[], format_block_content=False))
        torch.cuda.synchronize()
        inferred = time.perf_counter()
        if len(pages) != 1:
            raise RuntimeError(f"Expected one image result, got {len(pages)}")
        native = pages[0].json["res"]
        # JsonMixin converts numpy values; ensure the export contains only JSON values.
        native = json.loads(json.dumps(native, ensure_ascii=False, allow_nan=False))
        blocks = native["parsing_res_list"]
        text = "\n".join(block_to_text(block) for block in blocks)
        finished = time.perf_counter()
        truncated = any(call["truncated"] for call in calls)
        return {
            "text": text, "raw_response": "\n".join(call["raw_text"] for call in calls),
            "native_result": native, "generation_calls": calls,
            "lines": [{"text": block_to_text(block), "label": block["block_label"],
                       "bbox": block["block_bbox"], "confidence": None} for block in blocks],
            "layout": {"method": "native_paddleocr_vl_1_6", "blocks": blocks,
                       "region_count": len(blocks)},
            "image_size": list(image.size),
            "input_tokens": sum(call["input_tokens"] for call in calls),
            "generated_tokens": sum(call["generated_tokens"] for call in calls),
            "truncated": truncated, "finish_reason": "length" if truncated else "eos",
            "gpu_memory": {"peak_allocated_bytes": int(torch.cuda.max_memory_allocated()),
                           "peak_reserved_bytes": int(torch.cuda.max_memory_reserved())},
            "timing_seconds": {"decode": decoded - started,
                               "native_pipeline": inferred - decoded,
                               "text_assembly": finished - inferred,
                               "total": finished - started},
        }


## 4. Warm-up và nhận dạng

Warm-up dùng ảnh đầu tiên và giới hạn 64 token mỗi vùng; kết quả warm-up không được chấm.
Mỗi ảnh có log `START`, giai đoạn và số token mỗi 30 giây. Số token tăng cho biết vòng sinh đang tiến triển.
Giữ nguyên cấu hình khi tiếp tục: chạy lại ô benchmark sẽ bỏ qua các ID đã có checkpoint.
Lỗi đã lưu được giữ lại để báo cáo; dùng tên lượt mới nếu cần chạy lại toàn bộ.

`TOKEN LIMIT` nghĩa là văn bản có thể chưa đủ; bản chép dở vẫn tham gia CER/WER.
Khi đổi prompt, checkpoint, giới hạn ảnh hoặc token, đổi `RUN_NAME` để không trộn cấu hình.
Nếu cần dừng, bấm Stop và chạy ô tải ZIP. Không Disconnect/Delete runtime trước khi lưu kết quả.

Paddle dùng ngân sách **4.096 token/vùng**, tổng token/trang có thể lớn hơn 4.096.
Log token tính cộng dồn các vùng; phần warm-up cũng chạy layout toàn ảnh.


In [ ]:
print("GPU:", torch.cuda.get_device_name(0), flush=True)
print("Output:", OUTPUT_DIR, flush=True)
print("max_new_tokens:", MAX_NEW_TOKENS, "| max_pixels:", MAX_PIXELS, flush=True)
print("Log every 30 seconds during each image; completed images are resumed from checkpoints.",
      flush=True)
torch.cuda.synchronize()
warmup_started = time.perf_counter()
warmup_preview = predict_image(samples[0], max_new_tokens=WARMUP_NEW_TOKENS)
torch.cuda.synchronize()
warmup_seconds = time.perf_counter() - warmup_started
print("Warm-up:", round(warmup_seconds, 2), "seconds")
print("Preview (not scored):", warmup_preview["text"][:250])
del warmup_preview
gc.collect()
torch.cuda.empty_cache()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
atomic_json(OUTPUT_DIR / "manifest.json", manifest)
metadata = run_benchmark(
    samples, OUTPUT_DIR, identity,
    {"initialization_seconds": initialization_seconds, "warmup_seconds": warmup_seconds,
     "progress_monitor": "30s heartbeat + token observer; does not change generation settings"},
    predict_image)
print(json.dumps({k: metadata.get(k) for k in (
    "status", "completed_images", "failed_images", "pending_images", "truncated_images",
    "wall_seconds", "latency_p50_seconds", "latency_p95_seconds",
    "peak_gpu_allocated_bytes")}, indent=2))


### Kiểm tra khi nghi bị dừng

Chỉ chạy ô dưới sau khi đã dừng ô benchmark; một ô Python mới thường phải chờ ô đang chạy.
Đây là kiểm tra trạng thái GPU và checkpoint, không tiếp tục inference.
Colab miễn phí có hạn mức GPU thay đổi; notebook không gọi API tính phí theo token.
Chỉ xem xét hạn mức khi Colab hiển thị thông báo tương ứng.
[FAQ Colab](https://research.google.com/colaboratory/faq.html#resource-limits).


In [ ]:
try:
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,utilization.gpu,memory.used,memory.total",
         "--format=csv"], check=False, timeout=15)
except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
    print("Cannot query GPU:", exc)
run_file = OUTPUT_DIR / "run.json"
if run_file.exists():
    snapshot = json.loads(run_file.read_text(encoding="utf-8"))
    print("Saved status:", {key: snapshot.get(key) for key in
          ("status", "completed_images", "failed_images", "pending_images", "truncated_images")})
    records = load_records(OUTPUT_DIR, samples, snapshot["run_fingerprint"])
    for sample in samples:
        row = records.get(sample["sample_id"])
        if row is not None:
            print(sample["sample_id"], row["status"],
                  "seconds=", round(row["timing_seconds"]["total"], 1),
                  "tokens=", row.get("generated_tokens"),
                  "truncated=", row.get("truncated"))
        else:
            print("First pending image:", sample["sample_id"])
            break
else:
    print("No saved run.json at", OUTPUT_DIR)


## 5. Tải kết quả về máy

ZIP chứa `outputs/<tên lượt chạy>/`, không chứa ảnh hay trọng số. Thời gian tải trọng số/ảnh và warm-up
không cộng vào latency. `run.json` ghi riêng khởi tạo/warm-up theo session; `wall_seconds` cộng thời gian
vòng xử lý các session, gồm ghi checkpoint. Bộ nhớ GPU là đỉnh PyTorch allocated/reserved, không phải
toàn bộ VRAM theo nvidia-smi; chưa đo đỉnh RAM CPU. CPU và GPU cần báo cáo riêng khi so sánh tốc độ.


In [ ]:
import shutil
from google.colab import files

if not (OUTPUT_DIR / "run.json").exists():
    raise RuntimeError("Chưa có lượt chạy để tải")
saved_run = json.loads((OUTPUT_DIR / "run.json").read_text(encoding="utf-8"))
saved_records = load_records(OUTPUT_DIR, samples, saved_run["run_fingerprint"])
write_artifacts(OUTPUT_DIR, samples, saved_records, saved_run)
if saved_run["pending_images"] or saved_run["failed_images"]:
    print("Lượt chạy còn thiếu/lỗi:", saved_run["pending_images"], saved_run["failed_images"])
if saved_run["truncated_images"]:
    print("Ảnh chạm giới hạn token:", saved_run["truncated_images"])
archive = shutil.make_archive(
    str(Path("/content") / RUN_NAME), "zip", root_dir=WORK_ROOT,
    base_dir="outputs/" + RUN_NAME)
print("ZIP:", archive)
files.download(archive)


## 6. Chấm trên máy local

Tại `D:\2tuan`, kích hoạt `.venv`, giải nén ZIP vào repo để có thư mục
`outputs/paddleocr_vl_1_6_colab_t4_fp16/`, rồi chạy:

```powershell
. .\.venv\Scripts\Activate.ps1
Expand-Archive -LiteralPath "$HOME\Downloads\paddleocr_vl_1_6_colab_t4_fp16.zip" -DestinationPath .
python scripts/ocr_baseline.py score --run outputs/paddleocr_vl_1_6_colab_t4_fp16 --output outputs/paddleocr_vl_1_6_colab_t4_fp16/scores_py312.json
```

Sửa đường dẫn ZIP theo vị trí tải thực tế. Nếu đã có thư mục kết quả hoặc file điểm cùng tên,
dùng tên mới để giữ tách biệt các lượt chạy. Lệnh chấm dùng **ground truth local đã duyệt**,
không dùng nhãn cũ trên GitHub. Không chạy `draft` để thay ground truth.

Notebook đã được kiểm tra cú pháp và định dạng xuất/chấm bằng kiểm thử local; chưa thực thi
inference trên T4 trong môi trường phát triển. Sau lượt đầu, kiểm tra GPU, lỗi và số ảnh bị cắt
trong `run.json` trước khi đưa số liệu vào báo cáo.
